# 💧 AquaSense AI — 04: Model Evaluation & Benchmark
**Project:** Intelligent Water Quality Assessment and Potability Prediction Using Explainable Machine Learning  

### Overview:
In this notebook, we evaluate all 8 trained models on the untouched test set across 7 performance metrics:
- Accuracy, Precision, Recall, F1 Score, ROC-AUC, Matthews Correlation Coefficient (MCC), and Log Loss.
We also generate confusion matrices, ROC curves, PR curves, and select the best model.


In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.loader import DataLoader
from src.data.preprocessor import WaterQualityPreprocessor
from src.models.trainer import ModelTrainer
from src.models.evaluator import ModelEvaluator
from src.models.calibrator import ModelCalibrator
from src.utils.visualization import set_plot_style

set_plot_style()
print("Evaluation modules loaded.")


## 1. Load Test Data & Models


In [ ]:
dl = DataLoader(data_path="../data/raw/water_potability.csv")
df = dl.load()
_, X_test, _, y_test = dl.split(df, test_size=0.20, random_state=42)

preprocessor = WaterQualityPreprocessor.load("../models/preprocessor.pkl")
X_test_proc = preprocessor.transform(X_test)

models = ModelTrainer.load_all(path="../models")
print(f"Loaded {len(models)} models.")


## 2. Multi-Metric Benchmark Table


In [ ]:
evaluator = ModelEvaluator()
results_df = evaluator.evaluate_all(models, X_test_proc, y_test)
results_df.sort_values(by='mcc', ascending=False)


## 3. Identify & Declare Best Model


In [ ]:
best_model_name = evaluator.get_best_model(results_df, metric='mcc')
print(f"🏆 Best Model Selected by MCC: {best_model_name}")
best_model = models[best_model_name]
ModelTrainer().save_single("best_model", best_model, path="../models")


## 4. Confusion Matrices (2x4 Grid)


In [ ]:
cm_fig = evaluator.plot_confusion_matrices(models, X_test_proc, y_test)
plt.show()


## 5. ROC & Precision-Recall Curves


In [ ]:
roc_fig = evaluator.plot_roc_curves(models, X_test_proc, y_test)
plt.show()

pr_fig = evaluator.plot_pr_curves(models, X_test_proc, y_test)
plt.show()


## 6. Model Comparison Bar Chart


In [ ]:
comp_fig = evaluator.plot_model_comparison(results_df)
plt.show()


## 7. Probability Calibration on Best Model


In [ ]:
calibrator = ModelCalibrator(method='isotonic')
calibrated_model = calibrator.calibrate(best_model, X_test_proc, y_test)
calibrator.save(calibrated_model, path="../models/calibrated_best_model.pkl")

cal_fig = calibrator.plot_calibration_curve(calibrated_model, best_model, X_test_proc, y_test)
plt.show()
